In [1]:
import pandas as pd
import warnings
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


# 1. 데이터 로딩
cust_df = pd.read_csv("../data/santander-customer-satisfaction/train.csv", encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)
 

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [2]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [3]:
# 2. 불균형 확인
print(cust_df['TARGET'].value_counts())
unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()
print('unsatisfied 비율은 {0:.2f}'.format((unsatisfied_cnt / total_cnt)))

TARGET
0    73012
1     3008
Name: count, dtype: int64
unsatisfied 비율은 0.04


In [4]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [5]:
# 3. 이상값 탐지(var3의 min: -999999)
cust_df.describe()
# 0인 컬럼 존재.
cust_df.describe().loc["min",:]

ID                              1.00
var3                      -999999.00
var15                           5.00
imp_ent_var16_ult1              0.00
imp_op_var39_comer_ult1         0.00
                             ...    
saldo_medio_var44_hace3         0.00
saldo_medio_var44_ult1          0.00
saldo_medio_var44_ult3          0.00
var38                        5163.75
TARGET                          0.00
Name: min, Length: 371, dtype: float64

In [6]:
# 4-1. 전처리(var3의 -999999 -> 2  & ID 제거)
cust_df["var3"] = cust_df["var3"].replace(-999999,2)
cust_df.drop("ID", axis=1, inplace=True)

In [7]:
# 일반 데이터와 레이블 분리
X_features = cust_df.iloc[:,:-1]
y_labels = cust_df.iloc[:,-1]

In [8]:
# 4-2. 전처리(분산 0인 컬럼 제거)
# 1. 각 컬럼의 분산 계산
stds = X_features.std()

# 2. 분산이 0인(즉, 표준편차가 0이거나 값이 모두 똑같은) 컬럼 이름 추출
zero_var_cols = stds[stds == 0].index.tolist()

print(f"분산이 0인 컬럼 개수: {len(zero_var_cols)}")
print(f"삭제할 컬럼들: {zero_var_cols}")

# 3. 해당 컬럼들 제거
X_features_clean = X_features.drop(columns=zero_var_cols)

print(f"정제 전 피처 shape: {X_features.shape}")
print(f"정제 후 피처 shape: {X_features_clean.shape}")

분산이 0인 컬럼 개수: 34
삭제할 컬럼들: ['ind_var2_0', 'ind_var2', 'ind_var27_0', 'ind_var28_0', 'ind_var28', 'ind_var27', 'ind_var41', 'ind_var46_0', 'ind_var46', 'num_var27_0', 'num_var28_0', 'num_var28', 'num_var27', 'num_var41', 'num_var46_0', 'num_var46', 'saldo_var28', 'saldo_var27', 'saldo_var41', 'saldo_var46', 'imp_amort_var18_hace3', 'imp_amort_var34_hace3', 'imp_reemb_var13_hace3', 'imp_reemb_var33_hace3', 'imp_trasp_var17_out_hace3', 'imp_trasp_var33_out_hace3', 'num_var2_0_ult1', 'num_var2_ult1', 'num_reemb_var13_hace3', 'num_reemb_var33_hace3', 'num_trasp_var17_out_hace3', 'num_trasp_var33_out_hace3', 'saldo_var2_ult1', 'saldo_medio_var13_medio_hace3']
정제 전 피처 shape: (76020, 369)
정제 후 피처 shape: (76020, 335)


In [9]:
# 값이 완전히 동일한 컬럼 중 뒤에 나온 것들의 이름을 반환
import numpy as np

def find_duplicate_columns(df):

    groups = {}
    for col in df.columns:
        v = df[col].values
        # 지문(sum/min/max)으로 후보를 먼저 좁힘
        key = (v.sum(), v.min(), v.max())
        groups.setdefault(key, []).append(col)

    dup = set()
    for cols in groups.values():
        if len(cols) < 2:
            continue
        for i in range(len(cols)):
            if cols[i] in dup:
                continue
            for j in range(i + 1, len(cols)):
                if cols[j] in dup:
                    continue
                if np.array_equal(df[cols[i]].values, df[cols[j]].values):
                    dup.add(cols[j])
    return sorted(dup)


dup_cols = find_duplicate_columns(X_features_clean)
print(f"중복 컬럼 개수: {len(dup_cols)}")
print(f"삭제할 컬럼들: {dup_cols}")

X_features_clean = X_features_clean.drop(columns=dup_cols)
print(f"정제 후 피처 shape: {X_features_clean.shape}")

중복 컬럼 개수: 29
삭제할 컬럼들: ['delta_num_reemb_var13_1y3', 'delta_num_reemb_var17_1y3', 'delta_num_reemb_var33_1y3', 'delta_num_trasp_var17_in_1y3', 'delta_num_trasp_var17_out_1y3', 'delta_num_trasp_var33_in_1y3', 'delta_num_trasp_var33_out_1y3', 'ind_var13_medio', 'ind_var18', 'ind_var25', 'ind_var26', 'ind_var29', 'ind_var29_0', 'ind_var32', 'ind_var34', 'ind_var37', 'ind_var39', 'num_var13_medio', 'num_var18', 'num_var25', 'num_var26', 'num_var29', 'num_var29_0', 'num_var32', 'num_var34', 'num_var37', 'num_var39', 'saldo_medio_var13_medio_ult1', 'saldo_var29']
정제 후 피처 shape: (76020, 306)


In [10]:
# 학습 데이터 / 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X_features_clean, y_labels,
                                                    test_size=0.2, random_state=42)

In [11]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 수정된 get_clf_eval() 함수
def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [12]:
from sklearn.preprocessing import Binarizer


def get_eval_by_threshold(y_test , pred_proba_c1, thresholds):
    # thresholds 리스트 객체내의 값을 차례로 iteration하면서 Evaluation 수행.
    for custom_threshold in thresholds:
        binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_c1)
        custom_predict = binarizer.transform(pred_proba_c1)
        print('임곗값:',custom_threshold)
        get_clf_eval(y_test , custom_predict, pred_proba_c1)

In [14]:
# GradientBoosting으로 학습
from sklearn.ensemble import GradientBoostingClassifier
import time

start_time = time.time()

gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train, y_train)

pred = gb_clf.predict(X_test)
pred_proba = gb_clf.predict_proba(X_test)[:, -1]

print('GBM 수행 시간: {0:.1f} 초'.format(time.time() - start_time))
get_clf_eval(y_test, pred, pred_proba)

thresholds = [0.35, 0.4, 0.45]
get_eval_by_threshold(y_test, pred_proba.reshape(-1,1), thresholds)

# GBM 수행 시간: 35.8 초
# 오차 행렬
# [[14588     9]
#  [  607     0]]
# 정확도: 0.9595, 정밀도: 0.0000, 재현율: 0.0000,    F1: 0.0000, AUC:0.8365
# 임곗값: 0.1
# 오차 행렬
# [[13197  1400]
#  [  310   297]]
# 정확도: 0.8875, 정밀도: 0.1750, 재현율: 0.4893,    F1: 0.2578, AUC:0.8365
# 임곗값: 0.2
# 오차 행렬
# [[14110   487]
#  [  466   141]]
# 정확도: 0.9373, 정밀도: 0.2245, 재현율: 0.2323,    F1: 0.2283, AUC:0.8365
# 임곗값: 0.3
# 오차 행렬
# [[14540    57]
#  [  587    20]]
# 정확도: 0.9576, 정밀도: 0.2597, 재현율: 0.0329,    F1: 0.0585, AUC:0.8365

GBM 수행 시간: 35.9 초
오차 행렬
[[14588     9]
 [  607     0]]
정확도: 0.9595, 정밀도: 0.0000, 재현율: 0.0000,    F1: 0.0000, AUC:0.8365
임곗값: 0.35
오차 행렬
[[14581    16]
 [  603     4]]
정확도: 0.9593, 정밀도: 0.2000, 재현율: 0.0066,    F1: 0.0128, AUC:0.8365
임곗값: 0.4
오차 행렬
[[14584    13]
 [  606     1]]
정확도: 0.9593, 정밀도: 0.0714, 재현율: 0.0016,    F1: 0.0032, AUC:0.8365
임곗값: 0.45
오차 행렬
[[14588     9]
 [  607     0]]
정확도: 0.9595, 정밀도: 0.0000, 재현율: 0.0000,    F1: 0.0000, AUC:0.8365


In [15]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8]
}

# GradientBoostingClassifier 객체 생성 후 GridSearchCV 수행
gb_clf = GradientBoostingClassifier(random_state=42)
grid_cv = GridSearchCV(gb_clf, param_grid=params, scoring='roc_auc', cv=2, n_jobs=-1)
grid_cv.fit(X_train, y_train)

print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
print('최고 AUC: {0:.4f}'.format(grid_cv.best_score_))

최적 하이퍼 파라미터:
 {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
최고 AUC: 0.8307


In [16]:
# GridSearchCV로 찾은 최적 하이퍼 파라미터를 기반으로 n_estimators를 늘려 최종 학습
best_params = grid_cv.best_params_

gb_clf = GradientBoostingClassifier(
    random_state=42,
    n_estimators=300,
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    subsample=best_params['subsample'],
)
gb_clf.fit(X_train, y_train)

pred = gb_clf.predict(X_test)
pred_proba = gb_clf.predict_proba(X_test)[:, -1]

get_clf_eval(y_test, pred, pred_proba)

thresholds = [0.01, 0.05, 0.1, 0.2, 0.3, 0.33, 0.36, 0.39]
get_eval_by_threshold(y_test, pred_proba.reshape(-1,1), thresholds)

오차 행렬
[[14589     8]
 [  603     4]]
정확도: 0.9598, 정밀도: 0.3333, 재현율: 0.0066,    F1: 0.0129, AUC:0.8386
임곗값: 0.01
오차 행렬
[[6323 8274]
 [  35  572]]
정확도: 0.4535, 정밀도: 0.0647, 재현율: 0.9423,    F1: 0.1210, AUC:0.8386
임곗값: 0.05
오차 행렬
[[12284  2313]
 [  216   391]]
정확도: 0.8337, 정밀도: 0.1446, 재현율: 0.6442,    F1: 0.2362, AUC:0.8386
임곗값: 0.1
오차 행렬
[[13239  1358]
 [  301   306]]
정확도: 0.8909, 정밀도: 0.1839, 재현율: 0.5041,    F1: 0.2695, AUC:0.8386
임곗값: 0.2
오차 행렬
[[14070   527]
 [  463   144]]
정확도: 0.9349, 정밀도: 0.2146, 재현율: 0.2372,    F1: 0.2254, AUC:0.8386
임곗값: 0.3
오차 행렬
[[14510    87]
 [  575    32]]
정확도: 0.9565, 정밀도: 0.2689, 재현율: 0.0527,    F1: 0.0882, AUC:0.8386
임곗값: 0.33
오차 행렬
[[14548    49]
 [  587    20]]
정확도: 0.9582, 정밀도: 0.2899, 재현율: 0.0329,    F1: 0.0592, AUC:0.8386
임곗값: 0.36
오차 행렬
[[14567    30]
 [  595    12]]
정확도: 0.9589, 정밀도: 0.2857, 재현율: 0.0198,    F1: 0.0370, AUC:0.8386
임곗값: 0.39
오차 행렬
[[14574    23]
 [  597    10]]
정확도: 0.9592, 정밀도: 0.3030, 재현율: 0.0165,    F1: 0.0312, AUC:0.8386
